# Pseudobulk Differential Expression of Macrophage States Across Treatment Stages
PyDESeq2 with `~ smoking + stage`, testing pairwise stage contrasts. DEG counts, gene overlap across consecutive comparisons, per sample expression of selected genes and ranked log2FC plots are generated from the saved result tables.

In [ ]:
# !pip install --quiet scanpy decoupler pydeseq2 adjustText
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid

In [ ]:
import os
from collections import defaultdict

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
from adjustText import adjust_text
from pydeseq2.dds import DefaultInference, DeseqDataSet
from pydeseq2.ds import DeseqStats

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad"

deg_output_dir = "results/DEG"
figures_output_dir = "figures"

counts_layer = "raw_counts"
state_col = "cell_state"
stage_col = "recist_v2"
sample_col = "ID_2"
covariate = "smoking_status_grp"
exclude_ids = ["R008", "R011", "D001", "D002", "D003"]   # matched on obs["ID"]: liver biopsies and D samples

# comparisons: name -> (reference, test, states tested, file suffix); log2FC > 0 = higher in test
comparisons = {
    "AN_vs_TN": ("0_adj_normal", "1_treatment_naive",
                 ["Macrophage INHBAhigh", "Macrophage CD36high", "Macrophage STAB1high"], "naive_vs_adj_normal"),
    "TN_vs_OT": ("1_treatment_naive", "3_on_treatment",
                 ["Macrophage CD36high", "Macrophage STAB1high"], "on_treatment_vs_naive"),
    "TN_vs_R": ("1_treatment_naive", "4_TKI_resistance",
                ["Macrophage CD36high", "Macrophage STAB1high"], "TKIR_vs_naive"),
    "OT_vs_R": ("3_on_treatment", "4_TKI_resistance",
                ["Macrophage CD36high", "Macrophage STAB1high"], "TKIR_vs_on_treatment"),
}

# pseudobulk sample QC and gene filtering
min_cells_per_sample = 10
min_counts_per_sample = 500
filter_by_expr_kwargs = dict(min_count=10, min_total_count=15, large_n=10, min_prop=0.7)
filter_by_prop_kwargs = dict(min_prop=0.1, min_smpls=2)
n_cpus = 8

# DEG thresholds
padj_threshold = 0.05
lfc_threshold = 1

# consecutive comparisons used for DEG counts / overlap (label shown on plots)
steps = {"AN_vs_TN": "Normal→Tumour", "TN_vs_OT": "Tumour→On-treatment", "OT_vs_R": "On-treatment→Resistance"}
state_plot_order = ["Macrophage STAB1high", "Macrophage CD36high", "Macrophage INHBAhigh"]
deg_size_scale = 7

stage_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
stage_labels = {"0_adj_normal": "Adjacent\nNormal", "1_treatment_naive": "Treatment\nNaive",
                "3_on_treatment": "On\nTreatment", "4_TKI_resistance": "TKI\nRes"}
stage_colors = {"0_adj_normal": "#d86464", "1_treatment_naive": "#f4c9c9",
                "3_on_treatment": "#af8b8b", "4_TKI_resistance": "#636262"}

# Supplementary Fig. 5: per-sample log2CPM, statistics from one comparison
supp_fig5 = {"state": "Macrophage CD36high", "comparison": "TN_vs_OT",
             "genes": ["FGD6", "LIPA", "PNPLA3", "MGLL", "SCARB1", "APOO"]}

# ranked log2FC plots with curated labels
ranked_plots = [
    {"state": "Macrophage INHBAhigh", "comparison": "AN_vs_TN", "title": "Macrophage INHBAhigh — Naive vs Adj normal",
     "genes": ["CXCL8", "CSF1", "MAFF", "RBMS3", "IFITM10", "OASL", "FABP4", "RND3", "NLRP3", "CR1", "JUN", "SPTLC3",
               "STON2", "LRRK2", "CLEC2D", "BNC2", "PLAC8", "CD84", "NOTCH2", "CD52", "IL1B", "IL1A"]},
    {"state": "Macrophage CD36high", "comparison": "TN_vs_OT", "title": "Macrophage CD36high — On-treatment vs Naive",
     "genes": ["EREG", "ST6GALNAC3", "PDK4", "FLT1", "THBS1", "IL1RN", "NLRP3", "STON2", "MGLL", "KCNJ5", "FGD6",
               "APOO", "PIK3R3", "PNPLA3", "LIPA", "SAMD4A", "RAD32"]},
]

for d in (deg_output_dir, f"{deg_output_dir}/overlap", f"{deg_output_dir}/plots", figures_output_dir):
    os.makedirs(d, exist_ok=True)
inference = DefaultInference(n_cpus=n_cpus)

def result_path(state, comparison):
    return f"{deg_output_dir}/deseq2_{state}_{comparisons[comparison][3]}.csv"

## 1. Load and exclude samples

In [ ]:
adata = sc.read(input_path)
adata = adata[~adata.obs["ID"].isin(exclude_ids)].copy()
if counts_layer in adata.layers:
    adata.X = adata.layers[counts_layer].copy()
sample = adata.X[:1000].toarray() if sp.issparse(adata.X) else np.asarray(adata.X[:1000])
assert np.allclose(sample, np.round(sample)) and sample.min() >= 0, "pseudobulk needs raw counts"

states_used = sorted({s for _, _, states, _ in comparisons.values() for s in states})
print("cells per state x stage")
display(pd.crosstab(adata.obs[state_col], adata.obs[stage_col]).loc[states_used])
print("samples (cells >= min_cells_per_sample) per state x stage")
per_sample = adata.obs.groupby([state_col, stage_col, sample_col], observed=True).size()
display((per_sample >= min_cells_per_sample).groupby(level=[0, 1]).sum().unstack(fill_value=0).loc[states_used])

## 2. Pseudobulk DESeq2 per state and comparison

Samples with < `min_cells_per_sample` cells or < `min_counts_per_sample` counts are
dropped; genes are filtered with `filter_by_expr` and `filter_by_prop` on raw counts.
Design `~ covariate + stage`; Wald test of test vs reference stage.

In [ ]:
def run_deseq(state, comparison):
    ref, test, _, _ = comparisons[comparison]
    sub = adata[(adata.obs[state_col] == state) & adata.obs[stage_col].isin([ref, test])].copy()
    pdata = dc.pp.pseudobulk(adata=sub, sample_col=sample_col, groups_col=state_col, mode="sum")
    n_before = pdata.n_obs
    dc.pp.filter_samples(pdata, min_cells=min_cells_per_sample, min_counts=min_counts_per_sample)
    pdata.obs[stage_col] = pd.Categorical(pdata.obs[stage_col].astype(str), categories=[ref, test])
    pdata.obs[covariate] = pdata.obs[covariate].astype(str)

    design_table = pd.crosstab(pdata.obs[stage_col], pdata.obs[covariate])
    dc.pp.filter_by_expr(adata=pdata, group=stage_col, **filter_by_expr_kwargs)
    dc.pp.filter_by_prop(adata=pdata, **filter_by_prop_kwargs)
    print(f"{state} | {comparison}: {sub.n_obs} cells, samples {n_before} -> {pdata.n_obs}, genes {pdata.n_vars}")
    print(design_table.to_string(), "\n")

    dds = DeseqDataSet(adata=pdata, design=f"~{covariate} + {stage_col}", refit_cooks=True,
                       inference=inference, quiet=True)
    dds.deseq2()
    stats = DeseqStats(dds, contrast=[stage_col, test, ref], inference=inference, quiet=True)
    stats.summary()
    res = stats.results_df
    res.to_csv(result_path(state, comparison))

    fig = dc.pl.volcano(res, x="log2FoldChange", y="padj", top=20, thr_stat=lfc_threshold, thr_sign=padj_threshold,
                        color_pos=stage_colors[test], color_neg=stage_colors[ref], return_fig=True)
    fig.savefig(f"{figures_output_dir}/volcano_{state}_{comparisons[comparison][3]}.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return res, design_table

results, design_tables = {}, {}
for comparison, (_, _, states, _) in comparisons.items():
    for state in states:
        results[(state, comparison)], design_tables[(state, comparison)] = run_deseq(state, comparison)

## 3. DEG counts

Counted from the result tables (padj < `padj_threshold`, |log2FC| ≥ `lfc_threshold`),
for every state × comparison that was run.

In [ ]:
def deg_sets(res):
    sig = res[(res["padj"] < padj_threshold)]
    return (set(sig.index[sig["log2FoldChange"] >= lfc_threshold]),
            set(sig.index[sig["log2FoldChange"] <= -lfc_threshold]))

rows = []
for (state, comparison), res in results.items():
    up, down = deg_sets(res)
    rows.append({"state": state, "comparison": comparison, "n_samples": int(design_tables[(state, comparison)].values.sum()),
                 "n_genes_tested": int(res["padj"].notna().sum()), "Up": len(up), "Down": len(down)})
deg_counts = pd.DataFrame(rows)
deg_counts.to_csv(f"{deg_output_dir}/deg_counts_summary.csv", index=False)
deg_counts

In [ ]:
def deg_dotplot(counts, step_names, save_path, figsize):
    step_x = {name: i for i, name in enumerate(step_names)}
    states = [s for s in state_plot_order if s in set(counts["state"])]
    offset = {"Up": 0.18, "Down": -0.18}
    colors = {"Up": "darkorange", "Down": "#4A90E2"}
    fig, ax = plt.subplots(figsize=figsize)
    for direction in ("Up", "Down"):
        d = counts[counts["comparison"].isin(step_names)]
        ax.scatter(d["comparison"].map(step_x), d["state"].map({s: i for i, s in enumerate(states)}) + offset[direction],
                   s=d[direction] * deg_size_scale, c=colors[direction], edgecolors="black", linewidths=0.6,
                   alpha=0.85, clip_on=False)
    ax.set_xticks(range(len(step_names)), [steps[s] for s in step_names], rotation=30, ha="right")
    ax.set_yticks(range(len(states)), states)
    ax.set_xlim(-0.6, len(step_names) - 0.4)
    ax.set_ylim(-0.5, len(states) - 0.5)
    ax.set_xlabel("Stage comparison")
    ax.set_title("Macrophage DEG counts", weight="bold")
    ax.grid(True, axis="x", alpha=0.3)
    for side in ("right", "top"):
        ax.spines[side].set_visible(False)
    size_handles = [ax.scatter([], [], s=v * deg_size_scale, color="lightgray", edgecolors="black") for v in (5, 20, 50, 100)]
    leg = ax.legend(size_handles, [f"{v} DEGs" for v in (5, 20, 50, 100)], title="DEG count",
                    loc="upper center", bbox_to_anchor=(0.5, -0.55), ncol=2, frameon=False, labelspacing=1.2)
    ax.add_artist(leg)
    dir_handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors[k], markeredgecolor="black",
                              markersize=9, label=f"{k}regulated") for k in ("Up", "Down")]
    ax.legend(handles=dir_handles, title="Direction", loc="upper center", bbox_to_anchor=(0.5, -1.0), ncol=2, frameon=False)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

deg_dotplot(deg_counts, list(steps), f"{deg_output_dir}/plots/macrophage_deg_up_down_dotplot.pdf", (3.5, 4))
deg_dotplot(deg_counts, ["TN_vs_OT", "OT_vs_R"], f"{deg_output_dir}/plots/new_macrophage_deg_up_down_dotplot.pdf", (3, 3))

## 4. Gene overlap across consecutive comparisons

For each state and direction, each DEG is labelled by the comparisons in which it is significant. These are overlaps between independent pairwise contrasts in different patients.

In [ ]:
step_codes = {"AN_vs_TN": "N→T", "TN_vs_OT": "T→O", "OT_vs_R": "O→R"}
overlap_rows = []
for state in state_plot_order:
    available = [s for s in steps if (state, s) in results]
    if len(available) < 2:
        print(f"{state}: only {available} available; overlap skipped")
        continue
    for direction, idx in (("Up", 0), ("Down", 1)):
        sets = {s: deg_sets(results[(state, s)])[idx] for s in available}
        matrix = pd.DataFrame(0, index=available, columns=available)
        for a in available:
            for b in available:
                matrix.loc[a, b] = len(sets[a] & sets[b])
        matrix.index = matrix.columns = [step_codes[s] for s in available]
        matrix.to_csv(f"{deg_output_dir}/overlap/{state}_{direction}_overlap_matrix.csv")
        for gene in set().union(*sets.values()):
            pattern = " | ".join(step_codes[s] if gene in sets[s] else "—" for s in available)
            overlap_rows.append({"state": state, "direction": direction, "gene": gene, "pattern": pattern,
                                 "n_comparisons": sum(gene in sets[s] for s in available)})

        fig, ax = plt.subplots(figsize=(4, 3.5))
        sns.heatmap(matrix, annot=True, fmt="d", cmap="Reds" if direction == "Up" else "Blues", linewidths=1,
                    linecolor="white", square=True, cbar_kws={"label": f"shared {direction.lower()} DEGs"}, ax=ax)
        ax.set_title(f"{state}: {direction} DEG overlap", fontsize=10)
        plt.tight_layout()
        plt.savefig(f"{deg_output_dir}/plots/{state}_{direction}_overlap_matrix.pdf", dpi=300)
        plt.show()

overlap = pd.DataFrame(overlap_rows)
if len(overlap):
    overlap["program_type"] = overlap["n_comparisons"].map({1: "Stage-specific", 2: "Shared (2)", 3: "Shared (3)"})
    overlap.to_csv(f"{deg_output_dir}/ALL_macrophage_DEG_overlap_genes.csv", index=False)
    display(overlap.groupby(["state", "direction", "pattern"]).size().rename("n_genes").reset_index())

## 5. Per-sample expression of selected genes

log2 CPM of pseudobulk samples for all four stages (same sample QC as above). The
bracket shows DESeq2 log2FC and FDR from only; the other
stages are shown for context and were not part of that model.

In [ ]:
state, comparison = supp_fig5["state"], supp_fig5["comparison"]
ref, test = comparisons[comparison][:2]
stats_df = results[(state, comparison)]

sub = adata[(adata.obs[state_col] == state) & adata.obs[stage_col].isin(stage_order)].copy()
pdata = dc.pp.pseudobulk(adata=sub, sample_col=sample_col, groups_col=state_col, mode="sum")
dc.pp.filter_samples(pdata, min_cells=min_cells_per_sample, min_counts=min_counts_per_sample)
counts = pd.DataFrame(pdata.X.toarray() if sp.issparse(pdata.X) else pdata.X, index=pdata.obs_names, columns=pdata.var_names)
log2cpm = np.log2(counts.div(counts.sum(axis=1), axis=0) * 1e6 + 1)
print(pdata.obs[stage_col].value_counts().reindex(stage_order).to_string())

for gene in supp_fig5["genes"]:
    if gene not in log2cpm.columns:
        print(f"{gene}: not in data")
        continue
    d = pd.DataFrame({"log2CPM": log2cpm[gene], stage_col: pdata.obs[stage_col].astype(str)})
    fig, ax = plt.subplots(figsize=(3.2, 4))
    sns.boxplot(data=d, x=stage_col, y="log2CPM", order=stage_order, color="lightgrey", width=0.6,
                fliersize=0, boxprops=dict(alpha=0.5), ax=ax)
    sns.stripplot(data=d, x=stage_col, y="log2CPM", order=stage_order, hue=stage_col, hue_order=stage_order,
                  palette=stage_colors, jitter=0.15, size=8, linewidth=0, legend=False, ax=ax)
    if gene in stats_df.index:
        row = stats_df.loc[gene]
        lo, hi = d["log2CPM"].min(), d["log2CPM"].max()
        span = (hi - lo) or 1
        x1, x2 = stage_order.index(ref), stage_order.index(test)
        y, h = hi + 0.10 * span, 0.08 * span
        ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color="black", lw=1.3)
        ax.text((x1 + x2) / 2, y + h, f"log2FC={row['log2FoldChange']:.2f}\nFDR={row['padj']:.2e}",
                ha="center", va="bottom", fontsize=10)
        ax.set_ylim(top=y + h + 0.20 * span)
    ax.set_xticks(range(len(stage_order)), [stage_labels[s] for s in stage_order], rotation=45, ha="right")
    ax.set(xlabel="", ylabel="log$_2$ CPM", title=gene)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{figures_output_dir}/{gene}_{state.split()[-1]}_log2CPM_all_stages.pdf", dpi=300, bbox_inches="tight")
    plt.show() # Supplementary Figure 5

## 6. Ranked log2FC plots

All tested genes ranked by log2FC; size ∝ |log2FC|, colour = padj. Labelled genes are
curated in config; genes not in the result table are reported.

In [ ]:
for spec in ranked_plots:
    df = results[(spec["state"], spec["comparison"])].dropna(subset=["log2FoldChange"])
    df = df.sort_values("log2FoldChange", ascending=False).reset_index(names="gene")
    df["rank"] = np.arange(1, len(df) + 1)
    missing = [g for g in spec["genes"] if g not in set(df["gene"])]
    if missing:
        print(f"{spec['title']}: not in results {missing}")

    fig = plt.figure(figsize=(6.5, 4))
    ax = fig.add_axes([0.07, 0.15, 0.68, 0.72])
    pts = ax.scatter(df["rank"], df["log2FoldChange"], s=df["log2FoldChange"].abs() * 30, c=df["padj"],
                     cmap="winter_r", alpha=0.95, edgecolors="none")
    texts = [ax.text(r["rank"], r["log2FoldChange"], rf"$\it{{{r['gene']}}}$", fontsize=12,
                     bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.6), zorder=5)
             for _, r in df[df["gene"].isin(spec["genes"])].iterrows()]
    adjust_text(texts, ax=ax, only_move={"points": "y", "text": "xy"},
                arrowprops=dict(arrowstyle="-", lw=0.8, color="black"))
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(spec["title"], fontsize=13, pad=8)
    ax.set_xlabel("Ranking of differentially expressed genes", fontsize=14)
    ax.set_ylabel("Log2FoldChange", fontsize=14)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin - 0.5, ymax + 0.9)
    cax = fig.add_axes([0.78, 0.15, 0.02, 0.72])
    fig.colorbar(pts, cax=cax, label="padj")
    tag = f"{spec['state'].split()[-1]}_{comparisons[spec['comparison']][3]}"
    plt.savefig(f"{figures_output_dir}/ranked_log2FC_{tag}.pdf", dpi=300, bbox_inches="tight")
    plt.show() #Figure 3